# Dataset Statistics/exploration (Safe to run, better run one time)

Try to understand the quality of the data better. 
In total five types of data:

    1. DMD patterns 
    2. Chromox real beam
    3. YAG real beam
    4. Chromox laser scan
    5. Yag laser scan

Beam data started from Wednesday (Chromox) (2025-11-19), then fiber state changed, then Friday and Saturday (Chromox, 2025-11-21 + 2025-11-22), Sunday (YAG, 2025-11-23), with laser scan also in Saturday and Sunday, plus DMD in the middle of sections (Many collected in 2025-11-20)


In [1]:
import os
from pathlib import Path
os.chdir(Path.cwd().parent)   # go one level up
print(os.getcwd())         

from xflow import SqlProvider, flow, consume, TransformRegistry as T, as_hook, compose_hooks
from xflow.utils import plot_image
import xflow.extensions.physics
from xflow.utils.io import scan_files, create_directory
from xflow.utils.sql import union_sqlite_db_tables, merge_sqlite_dbs
from xflow.extensions.physics.beam import extract_beam_parameters

from config_utils import load_config, detect_machine
from utils import *

from functools import partial
import pandas as pd
import numpy as np
import sqlite3
import json
from PIL import Image
from tqdm import tqdm

experiment_name = "CAE_validate_clear"  
machine = detect_machine() 
config = load_config(
    f"{experiment_name}.yaml",
    machine=machine
)

dataset = "Chromox_only" # "DMD_only" or "Chromox_only"

# if win, windows test in machine then auto switch to windows path, otherwise use mac path
if any(win in machine for win in ["win", "windows"]): # windows
    dirs = {
        "merged_db_path": "C:/Users/qiyuanxu/Desktop/clear_2025_dataset.db",
        "raw_db_dir": "C:/Users/qiyuanxu/Documents/DataHub/datasets",
        "final_merged_db_path": "C:/Users/qiyuanxu/Desktop/dataset_meta.db",
        "DMD_only": {
            "output_dataset_dir": "C:/Users/qiyuanxu/Desktop/CLEAR25_DMD/dataset/",
            "dataset_dir": "C:/Users/qiyuanxu/Desktop/CLEAR25_DMD/",
            "output_db_dir": "C:/Users/qiyuanxu/Desktop/CLEAR25_DMD/db/dataset_meta.db",
        },
        "Chromox_only": {
            "output_dataset_dir": "C:/Users/qiyuanxu/Desktop/CLEAR25_Chromox/dataset/",
            "dataset_dir": "C:/Users/qiyuanxu/Desktop/CLEAR25_Chromox/",
            "output_db_dir": "C:/Users/qiyuanxu/Desktop/CLEAR25_Chromox/db/dataset_meta.db",
        },
        "Yag_only": {
            "output_dataset_dir": "C:/Users/qiyuanxu/Desktop/CLEAR25_Yag/dataset/",
            "dataset_dir": "C:/Users/qiyuanxu/Desktop/CLEAR25_Yag/",
            "output_db_dir": "C:/Users/qiyuanxu/Desktop/CLEAR25_Yag/db/dataset_meta.db",
        },
    }

elif any(mac in machine for mac in ["mac", "darwin"]): # Mac
    dirs = {
        "merged_db_path": "/Users/andrewxu/Desktop/clear_2025_dataset.db",
        "raw_db_dir": "/Users/andrewxu/Documents/DataHub/datasets",
        "final_merged_db_path": "/Users/andrewxu/Desktop/dataset_meta.db",
        "DMD_only": {
            "output_dataset_dir": "/Users/andrewxu/Desktop/CLEAR25_DMD/dataset/",
            "dataset_dir": "/Users/andrewxu/Desktop/CLEAR25_DMD/",
            "output_db_dir": "/Users/andrewxu/Desktop/CLEAR25_DMD/db/dataset_meta.db",
        },
        "Chromox_only": {
            "output_dataset_dir": "/Users/andrewxu/Desktop/CLEAR25_Chromox/dataset/",
            "dataset_dir": "/Users/andrewxu/Desktop/CLEAR25_Chromox/",
            "output_db_dir": "/Users/andrewxu/Desktop/CLEAR25_Chromox/db/dataset_meta.db",
        }
    }

else:
    raise ValueError(f"Unsupported machine: {machine}")

c:\Users\qiyuanxu\Documents\GitHub\fiber-image-reconstruction-comparison
[config_utils] Using machine profile: win-qiyuanxu


# Scope 1 - DMD synthetic data and its corresponding Real data

Create such ready to use dataset for training and evaluation (could reverse the training testing logic)

In [ ]:
# ============================
# Merge entire CLEAR 2025 dataset in to a single database
# ============================
db_paths = scan_files(dirs["raw_db_dir"], extensions=[".db"], return_type="str")
merge_sqlite_dbs(db_paths, output_path=dirs["merged_db_path"], source_column="db_path")

# ============================
# Left join metadata into the big merged database to form a single complete table
# ============================
sql = """
SELECT
    d.*,
    c.experiment_description,
    c.image_source,
    c.image_device,
    c.fiber_config,
    c.camera_config,
    c.other_config
FROM mmf_dataset_metadata AS d
LEFT JOIN mmf_experiment_config AS c
  ON c.id = d.config_id
 AND c.db_path = d.db_path;
"""

with sqlite3.connect(str(dirs["merged_db_path"])) as con:
    tables_df = pd.read_sql_query(sql, con)

# optional: drop duplicate column names (e.g. both tables have "id", "db_path")
tables_df = tables_df.loc[:, ~tables_df.columns.duplicated()]
print(tables_df.shape)

In [ ]:
# ============================
# DMD hourly data collection statistics
# ============================
def hourly_counts(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate rows by hour and return counts sorted chronologically."""
    return (
        pd.to_datetime(df["create_time"], errors="coerce")
          .dt.strftime("%Y-%m-%d %H")
          .dropna()
          .value_counts()
          .rename_axis("hour")
          .reset_index(name="count")
          .sort_values("hour")
          .reset_index(drop=True)
    )

tables_df["other_config"] = tables_df["other_config"].map(
    lambda x: x if isinstance(x, dict)
    else json.loads(x) if isinstance(x, str) and x.strip().startswith("{")
    else None
)

mask = tables_df["other_config"].map(
    lambda d: (
        isinstance(d, dict)
        and d.get("dmd_config", {}).get("type") != "DummyDMD"
        and d.get("beam_settings") is None
    )
)
dmd_df = tables_df.loc[mask].copy()
print("Total rows kept:", int(mask.sum()))
out_1 = hourly_counts(dmd_df)

# ============================
# Chromox hourly data collection statistics
# ============================
mask = tables_df["beam_settings"].notna() & tables_df["image_device"].astype(str).str.contains("Chromox", na=False)
chromox_df = tables_df.loc[mask].copy()
print("Total rows kept:", int(mask.sum()))
out_2 = hourly_counts(chromox_df)

# ============================
# YAG hourly data collection statistics
# ============================
mask = tables_df["beam_settings"].notna() & tables_df["image_device"].astype(str).str.contains("YAG", na=False)
yag_df = tables_df.loc[mask].copy()
print("Total rows kept:", int(mask.sum()))
out_3 = hourly_counts(yag_df)

In [ ]:
import matplotlib.pyplot as plt

for df in (out_1, out_2, out_3):
    df["hour"] = pd.to_datetime(df["hour"])
    df.sort_values("hour", inplace=True)

# union of all active hours
hours = sorted(set(out_1["hour"]) | set(out_2["hour"]) | set(out_3["hour"]))

# align each dataframe to the same active-hour axis
a = out_1.set_index("hour").reindex(hours, fill_value=0)["count"]
b = out_2.set_index("hour").reindex(hours, fill_value=0)["count"]
c = out_3.set_index("hour").reindex(hours, fill_value=0)["count"]

x = np.arange(len(hours))
w = 0.45

fig, ax = plt.subplots(figsize=(14, 5))

# bars
ax.bar(x - w, a, width=w, label="DMD", color="tab:blue", alpha=0.8, zorder=2)
ax.bar(x,     b, width=w, label="Chromox", color="tab:orange", alpha=0.8, zorder=2)
ax.bar(x + w, c, width=w, label="YAG", color="tab:green", alpha=0.8, zorder=2)

# alternating day shading + day labels
days = pd.Series(hours).dt.date
day_change_idx = np.where(days != days.shift(1))[0]
ymax = max(a.max(), b.max(), c.max())

for i, start in enumerate(day_change_idx):
    end = day_change_idx[i + 1] if i + 1 < len(day_change_idx) else len(hours)

    if i % 2 == 0:
        ax.axvspan(start - 0.5, end - 0.5, color="grey", alpha=0.26, zorder=0)

    x_mid = (start + end - 1) / 2
    day_text = pd.to_datetime(hours[start]).strftime("%Y-%m-%d")
    ax.text(
        x_mid, ymax * 1.02, day_text,
        ha="center", va="bottom",
        fontsize=9, color="grey", alpha=1
    )

ax.set_xticks(x)
ax.set_xticklabels([h.strftime("%m-%d %H") for h in hours], rotation=45, ha="right")
ax.set_xlabel("Hour")
ax.set_ylabel("Count")
ax.set_title("Hourly count distribution")
ax.set_ylim(0, ymax * 1.12)   # leave space for day labels
ax.legend()

plt.tight_layout()
plt.show() 

# Merge/Compile target .db then based on the .db create the final dataset
First select the data we want, compile to a single .db, create a column with abs path to samples, build datapipeline and go though this .db, keep the structure and save as a new ready to train folder
1. DMD only validation
2. If 1 works, DMD training + Chromox validation
3. There's two setup of MMF, one before 2025-11-20, one after. We do one after first
4. Figure out crop position

In general the valid DMD + Chromox as one 

In [ ]:
core_col_mmf_dataset_metadata = ["image_id", "is_calibration", "batch", "purpose", "db_path", "image_path", "comments", "beam_settings",
                                 "create_time", "is_saturated_ground_truth", "is_saturated_fiber_output", "config_id"]
core_col_mmf_experiment_config = ["experiment_description", "image_source", "image_device", "camera_config", "other_config"]

"""
 '2025-11-19',                          # dataset 1
 '2025-11-20',
 '2025-11-20_DMD_for_Wednesday_1',      # dataset 1
 '2025-11-20_DMD_for_Wednesday_2',      # dataset 1
 '2025-11-21',
 '2025-11-21-morning',
 '2025-11-22',
 '2025-11-22-afternoon',
 '2025-11-22-morning'
"""

df_to_merge = [dmd_df] # dmd_df, chromox_df, yag_df

final_df = pd.concat(df_to_merge, ignore_index=True, copy=False)
final_df = final_df.loc[:, [c for c in core_col_mmf_dataset_metadata + core_col_mmf_experiment_config if c in final_df.columns]]
base = final_df["db_path"].fillna("").str.rsplit("/", n=2).str[0] + "/"
final_df["image_path"] = base + final_df["image_path"].fillna("").str.lstrip("/")
final_df["dataset"] = final_df["db_path"].astype(str).str.rsplit("/", n=3).str[-3]
final_df = final_df.drop(columns=["db_path"])

remove_datasets = ['2025-11-19','2025-11-20_DMD_for_Wednesday_1','2025-11-20_DMD_for_Wednesday_2']  # these set are before fiber change. might be corrupted.
final_df = final_df[~final_df["dataset"].isin(remove_datasets)].copy()
final_df

In [ ]:
# ============================
# Extract data sample beam parameters, just to find the cropping area
# ============================

def add_image_features(df, path_col, extractors, in_place=True):
    """
    extractors: {"new_col_1": fn1, "new_col_2": fn2, ...}
      each fn takes left-half image as ndarray and returns a value (scalar / tuple / list)
    """
    if not in_place:
        df = df.copy()

    results = {new_col: [] for new_col in extractors}

    for p in tqdm(df[path_col].tolist(), desc="images"):
        if p is None or (isinstance(p, float) and np.isnan(p)) or str(p).strip() == "":
            for new_col in results:
                results[new_col].append(np.nan)
            continue

        try:
            with Image.open(p) as im:
                w, h = im.size
                left_np = np.asarray(im.crop((0, 0, w // 2, h)))
            for new_col, fn in extractors.items():
                results[new_col].append(fn(left_np))
        except Exception:
            for new_col in results:
                results[new_col].append(np.nan)

    for new_col, vals in results.items():
        df[new_col] = vals

    return df

extract_gaussian = partial(extract_beam_parameters, method="gaussian")
extract_moments = partial(extract_beam_parameters, method="moments")

final_df = add_image_features(
    final_df,
    path_col="image_path",
    extractors={"gaussian_beam_params": extract_gaussian, "moments_beam_params": extract_moments},
    in_place=True,
)

final_df = final_df.applymap(lambda x: str(x) if isinstance(x, (dict, list, tuple)) else x)
with sqlite3.connect(dirs["final_merged_db_path"]) as conn:
    final_df.to_sql("mmf_dataset_metadata", conn, if_exists="replace", index=False)

In [ ]:
# ============================
# Plot beam centroids and crop square, Define the cropping area
# ============================

import ast
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

vec_col = "moments_beam_params"
ds_col  = "dataset"

W, H = 1920, 1200          # image size
R = 150                    # half-side length in pixels (distance from centroid to each edge)     230

def parse_vec4(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, (list, tuple, np.ndarray)):
        a = list(v)
    else:
        s = str(v).strip()
        try:
            a = ast.literal_eval(s)
        except Exception:
            s = s.strip("[]()")
            a = [t for t in s.replace(",", " ").split() if t]
    a = [float(x) for x in a]
    return a if len(a) >= 2 else None

def centroid_xy_norm(xy_series, W=None, H=None, R=None, verbose=True):
    """
    xy_series: Series of list-like [x, y, ...] in normalized coords.
    R: half-side length in px (crop size is 2R x 2R).
    Returns:
      (cx_norm, cy_norm), (cx_px, cy_px), ((tl_px), (br_px)), ((tl_int), (br_int_excl))
    """
    arr = np.array([v[:2] for v in xy_series.dropna().to_list()], dtype=float)
    if arr.size == 0:
        return None

    cx_norm, cy_norm = arr[:, 0].mean(), arr[:, 1].mean()
    cx_px,  cy_px  = cx_norm * W, cy_norm * H

    tl = (cx_px - R, cy_px - R)
    br = (cx_px + R, cy_px + R)

    # integer crop box with exact size (2R x 2R); br_int is EXCLUSIVE
    R_int = int(R)
    tl_int = (int(round(tl[0])), int(round(tl[1])))
    br_int = (tl_int[0] + 2*R_int, tl_int[1] + 2*R_int)

    if verbose:
        print(f"centroid (px): ({cx_px:.2f}, {cy_px:.2f})")
        print(f"square TL, BR (px): ({tl[0]:.2f}, {tl[1]:.2f}), ({br[0]:.2f}, {br[1]:.2f})")
        print(f"square TL, BR (int, size={2*R_int}x{2*R_int} px): {tl_int}, {br_int}")

    return (cx_norm, cy_norm), (cx_px, cy_px), (tl, br)


xy = final_df[vec_col].apply(parse_vec4)
mask = xy.notna() & final_df[ds_col].notna()

# normalized points
x_norm = xy[mask].apply(lambda a: a[0]).to_numpy()
y_norm = xy[mask].apply(lambda a: a[1]).to_numpy()
ds = pd.Categorical(final_df.loc[mask, ds_col].astype(str))

# centroid in normalized coords
c = centroid_xy_norm(xy[mask], W=W, H=H, R=R, verbose=True)
(cx_norm, cy_norm), (cx, cy), (tl, br) = c

# convert to pixel coords only for plotting
x = x_norm * W
y = y_norm * H
cx = cx_norm * W
cy = cy_norm * H

cmap = plt.get_cmap("tab10")
colors = [cmap(i % cmap.N) for i in ds.codes]

fig, ax = plt.subplots(figsize=(15, 15), dpi=120)
ax.scatter(x, y, s=1, c=colors, alpha=0.5, edgecolors="none")

# legend (one color per dataset)
for i, name in enumerate(ds.categories):
    ax.scatter([], [], s=30, color=cmap(i % cmap.N), label=name)
ax.legend(title=ds_col, loc="upper left", bbox_to_anchor=(1.02, 1.0))

# centroid (star) + square of half-side length R
ax.scatter([cx], [cy], marker="*", s=20)
ax.add_patch(Rectangle((cx - R, cy - R), 2*R, 2*R, fill=False, linewidth=2))

ax.set_xlim(0, W)
ax.set_ylim(0, H)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x (px)")
ax.set_ylabel("y (px)")
plt.tight_layout()
plt.show()


In [ ]:
# No need to rerun, just read the final merged database and continue analysis
table = "mmf_dataset_metadata"

with sqlite3.connect(dirs["final_merged_db_path"]) as conn:
    final_df = pd.read_sql_query(f'SELECT * FROM "{table}"', conn)

In [ ]:
# ============================
# Core dataset creation pipeline
# ============================

params = {
    # DMD only: [[742, 341], [1202, 801]]
    # Chromox only: [[828, 459], [1128, 759]]
    "crop_gt": [[742, 341], [1202, 801]],   
    "crop_fo": [[360, 0], [1560, 1200]], 
    "inp_size": (256, 256),
    "out_size": (256, 256)
}

resize_fo = (params["crop_gt"][1][0] - params["crop_gt"][0][0], params["crop_gt"][1][1] - params["crop_gt"][0][1])
image_paths = final_df["image_path"].tolist()

results = list(flow(
    [(p, p) for p in image_paths],
    [T.get("torch_load_image"), T.get("extract_stem")],
    [None, partial(T.get("add_parent_dir"), parent_dir=dirs[dataset]["output_dataset_dir"])], # CLEAR25_CHROMOX
    [None, partial(T.get("add_suffix"), suffix=".png")],
    # [None, T.get("debug_print")],
    # [T.get("torch_to_grayscale"), None],
    # [T.get("torch_remap_range"), None],
    
    [partial(T.get("torch_split_width"), swap=True), None],
    [
        partial(T.get("torch_crop_area"), points=params["crop_fo"]),
        partial(T.get("torch_crop_area"), points=params["crop_gt"]),
        None,
    ],
    [ 
        partial(T.get("torch_resize"), size=params["inp_size"], interpolation="bilinear"),     
        partial(T.get("torch_resize"), size=params["out_size"], interpolation="bilinear"),
        None,
    ],
    partial(T.get("reorder"), order=[1, 0, 2]),
    [consume(2, lambda imgs: T.get("join_image")(list(imgs), layout=(1, 2))), None],
    T.get("save_image"),  # Takes (image, full_path), saves, passes through
    
    progress=True,
    desc="Processing images",
    skip_errors=False,
))

# also save the metadata dataframe as .db, then it is ready for training on HPC/GPU
df_save = final_df.copy()
s = df_save["image_path"]
df_save["image_path"] = s.where(
    s.isna(),
    "dataset/" + s.str.replace(r"^.*[\\/]", "", regex=True)
)

create_directory(dirs[dataset]["output_db_dir"])
with sqlite3.connect(dirs[dataset]["output_db_dir"]) as conn:
    df_save.to_sql("mmf_dataset_metadata", conn, if_exists="replace", index=False)

## quick test of the created basis pipeline before HPC
Once the processed dataset is created, could directly run below (and the first config cell) cells without run anything else

In [ ]:
from xflow.extensions.physics.pipeline import CachedBasisPipeline, IndexCombinator
from xflow.data import build_transforms_from_config, PyTorchPipeline
from xflow.extensions.physics import pattern_gen

config = load_config(
    f"{experiment_name}.yaml",
    machine=machine
)

# training set (basis -> generative pipeline)
train_provider = SqlProvider(
    sources={"connection": dirs[dataset]["output_db_dir"], "sql": config["sql"]["train"]}, output_config={'list': "image_path"}
)
# evaluation set (validation + test, using real beam pattern loaded on the DMD)
eval_provider = SqlProvider(
    sources={"connection": dirs[dataset]["output_db_dir"], "sql": config["sql"]["eval"]}, output_config={'list': "image_path"}
)

background_provider = SqlProvider(
    sources={"connection": dirs[dataset]["output_db_dir"], "sql": config["sql"]["background"]}, output_config={'list': "image_path"}
)

# create data pipelines for ML training and evaluation
config["data"]["transforms"]["torch"].insert(0, {
    "name": "add_parent_dir",
    "params": {
        "parent_dir": dirs[dataset]["dataset_dir"]
    }
})
transforms = build_transforms_from_config(config["data"]["transforms"]["torch"])

canvas = pattern_gen.DynamicPatterns(256, 256)
canvas.set_postprocess_fns([T.get("remap_range"), partial(T.get("resize"), size=(32,32), interpolation="bilinear")])
canvas._distributions = [pattern_gen.StaticGaussianDistribution(canvas) for _ in range(100)]
canvas.set_threshold(10)
# (std_1=0.03, std_2=0.2, max_intensity=100, fade_rate=0.96, distribution='other'
stream = canvas.pattern_stream(std_1=0.02, std_2=0.08, max_intensity=100, fade_rate=0.98, distribution='other') 

# Option: for using runtime sample to background subtract
background_pipeline = PyTorchPipeline(
    background_provider, 
    transforms[:3],
).to_memory_dataset()
runtime_transforms = [partial(T.get("torch_subtract_tensor"), subtract_tensor=next(iter(background_pipeline))[0]), T.get("torch_clip_below_zero")]
# transforms[3:3] = runtime_transforms

# ======== random combinator using index + SGM ========
combinator = IndexCombinator(
    pattern_provider=stream,
    post_transforms= build_transforms_from_config(config["data"]["post_transforms"]["torch"]),
)

val_provider, test_provider = eval_provider.split(config["data"]["val_test_split"])
train_pipeline = CachedBasisPipeline(
    train_provider, 
    combinator=combinator, 
    transforms=transforms, 
    num_samples=1000, 
    seed=42, 
    eager=True,
)

val_pipeline = PyTorchPipeline(
    val_provider, 
    transforms[:-1],
).to_memory_dataset(config["data"]["dataset_ops"])   # testset data do not need thresholding since it is to remove stacking noise?

test_pipeline = PyTorchPipeline(
    test_provider, 
    transforms[:-1],
).to_memory_dataset(config["data"]["dataset_ops"])

In [ ]:
# Each dataset (train, val, test) takes some random samples for visualization and sanity check 
# Specifically for the training set, also print out the basis combination info
for sample in train_pipeline:
    record = combinator.last_record
    # for idx, coef in zip(record.indices, record.coefficients):
    #     basis_item = train_pipeline.get_basis(idx)
    #     print(f"  basis[{idx}] * {coef:.2f}")
    left_parts, right_parts = sample
    plot_image(left_parts[0], title="Input (Fiber Output)", figsize=(6,6))
    plot_image(right_parts[0], title="Ground Truth", figsize=(6,6))
    break

for left_parts, right_parts in val_pipeline:
    plot_image(left_parts[0], title="Input (Fiber Output)", figsize=(6,6))
    plot_image(right_parts[0], title="Ground Truth", figsize=(6,6))
    break
# for left_parts, right_parts in test_pipeline:
#     plot_image(left_parts[0], title="Input (Fiber Output)", figsize=(6,6))
#     plot_image(right_parts[0], title="Ground Truth", figsize=(6,6))
#     break

# TODO: index scale factor is not totally correct, since the peak intensity of basis is not exactly 1.0
# TODO: noise problem emerges, when stacking too many basis patterns together, need to investigate further

# Scope 2 - Real data only with data augmentation pipeline (super position)
Create such ready to use dataset for training and evaluation